# Data on the Line
*Turning Demand Insights into Balanced Production* · Case study · KoçDigital analytics team · 120 minutes · Forecast date: **31 December 2025**

| Minute | What we do | In this notebook |
|---|---|---|
| 0–10 | Read the case text | — |
| 10–45 | Forecast demand | Part 1 |
| 45 | Upload the forecast; first password announced | `TeamXX_forecast.csv` from Step 1.3 → “Minute-45 forecast” form |
| 45–57 | Production plan and takt | Part 2 |
| 57–90 | Balance the line; the jury comes round | Part 3 |
| 90–95 | Finishing the line design and the line slide | the answers in Part 5, the forecast and plan slide, the line slide |
| 95 | Second password announced | Part 4 |
| 95–115 | URGENT order, finish the URGENT slide | Part 4 and Part 5 |
| 115 | Final submission | three files → “Final submission” form |
| 115–120 | Short closing | — |

The session is 120 minutes in all, and nothing is accepted after minute 115. So don't leave the slides until the end: from Part 2 (production plan) onwards, one person should fill them in alongside the analysis (the forecast and plan slide by minute 57, the line slide by minute 95), leaving only the URGENT slide and final touches for after minute 95. Upload your files as soon as they're ready; if you upload again, the latest upload counts.

**How this notebook works**

- Each part opens with a short brief and then goes step by step.
- After the brief come code cells headed **READY**: data, metrics, checks and the submission files. Just run them; read them if you're curious, but you don't need to change anything.
- Cells marked `# TODO` are yours.
- Every part has a check cell that tells you whether you're on the right track. January is the worked example: for January the check shows you the correct values. For the other months, and for your design choices, you're on your own.
- The rules are in the messages in the case text. Working out what they mean is part of the job.
- AI assistants are allowed, but the jury may ask you to explain any line of code you hand in.

A suggestion for splitting the work: one person on the forecast, one on the plan and the line, and one on the slides from the start. You don't have to wait for the takt time to start on the line model; try it with placeholders such as `C, MIX = 85, {"ND100": 0.4, "ND200": 0.35, "ND300": 0.25}`.

### Getting started

**First, type in your team number.** In the first cell below, change `TEAM = "Team__"` to the number your instructor gave you, for example `TEAM = "Team03"`. Your submission files are named after it (`Team03_forecast.csv`, `Team03_submission.zip`), and you'll upload them to two forms: *Data on the Line – Minute-45 forecast* and *Data on the Line – Final submission*. Sign in to the forms with the same Google account you use for Colab.

If you've opened the notebook in Colab, the next cell downloads the data files from GitHub by itself; there's nothing to upload. Once you've run the cells you should see **"Data loaded ✓"**.

Colab clears its working files if a session sits idle for too long. To keep your work, save a copy of the notebook every now and then with *File → Save a copy in Drive*.

If you're working on your own computer with Jupyter, keep the notebook and the `data/` folder together.

In [ ]:
TEAM = "Team__"   # the team number your instructor gave you, for example "Team03"

In [ ]:
# READY · Start: libraries and data

import importlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
import zipfile
import zlib
from datetime import datetime
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# on Colab the data files come from GitHub automatically; on your own computer this is skipped
REPO_URL = "https://github.com/burcu-koca-kd/data-on-the-line.git"
CONTENT = Path("/content")
REPO = CONTENT / "data-on-the-line"
CASE_DIR = REPO / "en/2_Case"


def _clone():
    try:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(REPO)],
                       check=True, capture_output=True)
    except subprocess.CalledProcessError:
        print("✗ The data files couldn't be downloaded from GitHub. Check your internet "
              "connection, then open the notebook link again or rerun this cell.")
        raise RuntimeError("Data files not downloaded (see the message above).") from None


def _update():
    """Brings the downloaded copy up to date with GitHub; leaves release/ and submission/ alone."""
    git = ["git", "-C", str(REPO)]
    fetched = subprocess.run(git + ["fetch", "-q", "--depth", "1"], capture_output=True)
    if fetched.returncode == 0:
        subprocess.run(git + ["reset", "-q", "--hard", "origin/main"], capture_output=True)


def _uploaded_case():
    """If the repository has no case folder yet, looks for a 2_Case folder (or 2_Case.zip) uploaded
    to Colab or to Drive."""
    for archive in sorted(CONTENT.glob("2_Case*.zip")):
        with zipfile.ZipFile(archive) as z:
            z.extractall(CONTENT)
    for root in (CONTENT, CONTENT / "drive" / "MyDrive"):
        for folder in (root / "2_Case", *root.glob("*/2_Case"), root):
            if (folder / "data" / "tasks.csv").exists():
                return folder
    return None


if "google.colab" in sys.modules:
    # a copy downloaded earlier is brought up to date (skipped quietly when offline)
    if REPO.exists():
        _update()
    # if the old copy has no case folder, the repository is downloaded again
    if not CASE_DIR.exists():
        shutil.rmtree(REPO, ignore_errors=True)
        _clone()
    # if the case folder isn't in the repository yet, look for an uploaded copy
    if not CASE_DIR.exists():
        CASE_DIR = _uploaded_case()
        if CASE_DIR is None:
            print("✗ The case files don't seem to be published yet: the repository has no 2_Case "
                  "folder. Let your instructor know. If your instructor has handed out the 2_Case "
                  "folder: (1) drag 2_Case.zip into the Files panel on the left-hand side of "
                  "Colab, or (2) put the 2_Case folder in your Drive and connect Drive from the "
                  "Files panel; then run this cell again.")
            raise RuntimeError("Case folder not found (see the message above).")
        print(f"No case folder in the repository; using the uploaded copy: {CASE_DIR}")
    os.chdir(CASE_DIR)


def _install(*packages):
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f"pip install {' '.join(packages)} failed. End of the error message:")
        print(result.stderr[-1500:])


def _pulp_major():
    try:
        return int(version("pulp").split(".")[0])
    except PackageNotFoundError:
        return 0


def _forget_pulp():
    """Drops PuLP from memory so that the next import sees newly installed packages."""
    for name in [m for m in sys.modules if m == "pulp" or m.startswith("pulp.")]:
        del sys.modules[name]
    importlib.invalidate_caches()


# Colab has either no PuLP or PuLP 4 installed; the case is written for version 3
if not 2 <= _pulp_major() < 4:
    _install("pulp>=2.7,<4")
    _forget_pulp()
if _pulp_major() >= 4:
    print("⚠ PuLP 3 could not be installed and version 4 is still loaded. Restart the session "
          "(Runtime → Restart session) and run this cell again.")
import pulp

DATA, RELEASE, SUBMISSION = Path("data"), Path("release"), Path("submission")
UPLOAD = Path("upload")
SUBMISSION.mkdir(exist_ok=True)
# the files for the forms (TeamXX_...) are collected here
UPLOAD.mkdir(exist_ok=True)


def _team():
    """Returns TEAM as 'Team03'; warns and returns None if it hasn't been filled in."""
    name = str(globals().get("TEAM", "")).strip()
    found = (re.fullmatch(r"0*(\d{1,2})", name)
             or re.search(r"(?i)(?:team|tak[ıi]m)\s*[-_]?\s*0*(\d{1,2})(?!\d)", name))
    if not found or int(found.group(1)) == 0:
        print("⚠ Your team number isn't filled in: change TEAM = 'Team__' at the top of the "
              "notebook to your number (for example TEAM = 'Team03'), then run that cell and this "
              "one again. Otherwise your files are named TeamXX and your instructor may not be "
              "able to match them to your team.")
        return None
    return f"Team{int(found.group(1)):02d}"


PRODUCTS = ["ND100", "ND200", "ND300"]
MONTHS = ["2026-01", "2026-02", "2026-03"]

history = pd.read_csv(DATA / "demand_history.csv").fillna({"Calendar_Note": ""})
plan_q1 = pd.read_csv(DATA / "promo_price_plan.csv", dtype={"Promo_Status": str})
plan_q1 = plan_q1.fillna({"Calendar_Note": "", "Promo_Status": ""})
calendar = pd.read_csv(DATA / "production_calendar.csv")
products = pd.read_csv(DATA / "products.csv")
tasks = pd.read_csv(DATA / "tasks.csv")
precedence = pd.read_csv(DATA / "precedence.csv")
current = pd.read_csv(DATA / "current_line.csv")

TASKS = list(tasks.Task)
# TIME[task][product] in seconds
TIME = {r.Task: {p: getattr(r, f"{p}_sec") for p in PRODUCTS} for r in tasks.itertuples()}
GROUP = dict(zip(tasks.Task, tasks.Process_Group))                 # process group of each task
# (a, b): a's station must not come after b's (same station is fine)
ARCS = list(zip(precedence.Predecessor, precedence.Successor))
CURRENT_LINE = dict(zip(current.Task, current.Station))            # today's layout {task: station}

# reference values for the check cells: January and the benchmark
NAIVE_WAPE, NAIVE_BIAS = 0.1070, -0.0864
JAN_AVAILABLE, JAN_PLANNED, JAN_TAKT = 1039500, 11180, 92.98


def fmt(x, decimals=0):
    """A number with thousands separators: 1,039,500 or 92.98."""
    return f"{x:,.{decimals}f}"


def pct(x, decimals=1, signed=False):
    """A share as a percentage: 10.7%; with signed=True, +10.7% or −8.6%."""
    sign = ("+" if x >= 0 else "−") if signed else ("−" if x < 0 else "")
    return f"{sign}{abs(x) * 100:.{decimals}f}%"


def show(table, decimals=2):
    """Displays a table with thousands separators: 1,039,500 and 92.98 (whole numbers without
    decimals)."""
    whole = [c for c in table.select_dtypes("float") if (table[c].dropna() % 1 == 0).all()]
    try:
        styled = table.style.format(precision=decimals, thousands=",")
        display(styled.format(precision=0, thousands=",", subset=whole))
    except (ImportError, AttributeError):          # pandas needs jinja2 for styled tables
        display(table)


def _as_dict(values):
    """A dictionary or a pandas Series as a plain dictionary; None stays None."""
    return None if values is None else dict(values)


def add_features(df):
    """Adds columns that models often need. Works for history and for plan_q1."""
    out = df.copy()
    period = pd.PeriodIndex(out.Month, freq="M")
    out["t"] = period.astype("int64")                              # month counter, for a trend
    out["month"] = period.month                                    # 1-12
    price = out["Avg_Price_EUR"] if "Avg_Price_EUR" in out else out["Planned_Price_EUR"]
    # log(price / list price): 0 means no discount
    out["discount"] = np.log(price / out["List_Price_EUR"])
    past = history.set_index(["Product", "Month"]).Demand_Units
    a_year_earlier = [str(pd.Period(m, freq="M") - 12) for m in out.Month]
    # demand of the same product twelve months earlier
    out["lag12"] = [past.get(key) for key in zip(out.Product, a_year_earlier)]
    return out


history, plan_q1 = add_features(history), add_features(plan_q1)

In [ ]:
# READY · Solver: CBC first, HiGHS if CBC won't start

def _solver_works(solver):
    try:
        test = pulp.LpProblem("test", pulp.LpMinimize)
        x = pulp.LpVariable("x", cat="Binary")
        test += x
        test += x >= 1
        test.solve(solver)
        return pulp.LpStatus[test.status] == "Optimal"
    except Exception:
        return False


def _pick_solver():
    global pulp
    # gapRel=0 for both CBC and HiGHS: solve to proven optimality (by default they may stop within
    #   0.01%, which here can hide half a second of balance)
    options = {"msg": False, "timeLimit": 60, "gapRel": 0}
    if _solver_works(pulp.PULP_CBC_CMD(msg=False)):
        return pulp.PULP_CBC_CMD(**options), "CBC"
    # backup solver: installed only if CBC won't start (on some Macs, for example)
    if find_spec("highspy") is None:
        _install("highspy")
        _forget_pulp()
        import pulp
    if hasattr(pulp, "HiGHS") and _solver_works(pulp.HiGHS(msg=False)):
        # HiGHS presolve off: with it on, HiGHS sometimes misses the best balance on this model
        return pulp.HiGHS(**options, presolve="off"), "HiGHS"
    return None, None


SOLVER, SOLVER_NAME = _pick_solver()

if SOLVER:
    print(f"Data loaded ✓ | solver: {SOLVER_NAME}")
else:
    print("Data loaded, but the solver won't start. The easy fix is to open the notebook in "
          "Colab. To work on your own Mac, run 'pip install highspy' in Terminal and restart the "
          "kernel.")
if _team():
    print(f"Team: {_team()}")

In [ ]:
print("Demand history (last rows):")
show(history.tail(6), decimals=3)
print("January–March 2026 plan:")
show(plan_q1, decimals=3)
demand = history.pivot(index="Month", columns="Product", values="Demand_Units")
demand.plot(figsize=(11, 3), title="Monthly demand (units)")
plt.show()

## 1 · Demand forecast (minutes 10–45)

**Goal:** forecast January–March 2026 demand for each product as of 31 December 2025, and use a backtest to show why your model deserves to be trusted.

**What you have**
- `history`: monthly demand for 2021–2025, with the promotion flag, prices, working days and a calendar note.
- `plan_q1`: the January–March 2026 plan Kerem sent; each promotion's approval status is in the `Promo_Status` column.
- Ready-made columns: `t` (month counter), `month`, `discount` (log of price over list price) and `lag12` (demand twelve months earlier).
- Ready-made functions: `wape()`, `bias()` and `seasonal_naive()`, the benchmark every model has to beat. If you change the price or promotion columns of a table, `add_features()` recalculates the ready-made columns.

In [ ]:
# READY · Forecast tools

def wape(actual, forecast):
    """WAPE: total absolute error over total actual demand. Lower is better."""
    actual, forecast = np.asarray(actual, float), np.asarray(forecast, float)
    return np.abs(actual - forecast).sum() / actual.sum()


def bias(actual, forecast):
    """Bias: (total forecast − total actual) / total actual. Positive means over-forecasting,
    negative under-forecasting."""
    actual, forecast = np.asarray(actual, float), np.asarray(forecast, float)
    return (forecast - actual).sum() / actual.sum()


def seasonal_naive(train, test):
    """Benchmark: the same month last year."""
    return test["lag12"].to_numpy(float)


def check_backtest(backtest):
    """Runs your backtest on the benchmark and compares it with the correct result."""
    problems = set()

    def probe(train, test):
        if len(train) and train.t.max() >= test.t.min():
            problems.add("future")
        if "Demand_Units" in test.columns:
            problems.add("actuals")
        return seasonal_naive(train, test)

    bt = backtest(probe)
    if len(bt) == 0:
        print("The backtest returns nothing yet.")
    elif "future" in problems:
        print("✗ At some origins the model also sees data from after the months it is forecasting.")
    elif "actuals" in problems:
        print("✗ test still has the Demand_Units column, so the model can see the actuals. Drop "
              "it before calling the model: test.drop(columns='Demand_Units').")
    else:
        w, b = wape(bt.Demand_Units, bt.Forecast), bias(bt.Demand_Units, bt.Forecast)
        if len(bt) == 72 and abs(w - NAIVE_WAPE) < 0.0005 and abs(b - NAIVE_BIAS) < 0.0005:
            print(f"✓ Your time split is correct: {len(bt)} forecasts, seasonal naive WAPE "
                  f"{pct(w)}, bias {pct(b, signed=True)}.")
        else:
            print("✗ For seasonal naive, a correct backtest gives 72 forecasts, WAPE "
                  f"{pct(NAIVE_WAPE)} and bias {pct(NAIVE_BIAS, signed=True)}. Yours: {len(bt)} "
                  f"forecasts, WAPE {pct(w)}, bias {pct(b, signed=True)}.")


def compare(models, backtest):
    """Backtests every model and shows WAPE, bias and bias by product."""
    rows = {}
    for name, model in models.items():
        bt = backtest(model)
        if len(bt) == 0:
            print("The backtest returns nothing yet.")
            return
        w = wape(bt.Demand_Units, bt.Forecast)
        rows[name] = {"WAPE": pct(w), "Bias": pct(bias(bt.Demand_Units, bt.Forecast), signed=True)}
        for p, g in bt.groupby("Product"):
            rows[name][f"Bias {p}"] = pct(bias(g.Demand_Units, g.Forecast), signed=True)
        if w < 0.01:
            print(f"⚠ {name}: WAPE {pct(w)}. An error this low usually means the model can see "
                  "the actuals; check for leakage.")
    display(pd.DataFrame.from_dict(rows, orient="index"))


def save_forecast(forecast, download=None):
    """Checks the format, rounds the units and writes submission/forecast.csv with the team number
    and the time it was saved. On Colab it downloads the file as TeamXX_forecast.csv, unless the
    forecast is still the benchmark's and download=True isn't given."""
    columns = ["Month", "Product", "Forecast_Units"]
    if not set(columns) <= set(forecast.columns):
        print("The forecast table needs the columns Month, Product and Forecast_Units; nothing "
              "was saved.")
        return
    out = forecast[columns].copy()
    units = pd.to_numeric(out.Forecast_Units, errors="coerce")
    keys = set(zip(out.Month.astype(str), out.Product.astype(str)))
    expected = {(m, p) for m in MONTHS for p in PRODUCTS}
    if len(out) != 9 or keys != expected or units.isna().any() or (units <= 0).any():
        print("The forecast needs exactly one row for each month of January–March 2026 and each "
              "product (9 rows in all), with positive values; nothing was saved.")
        return
    out["Forecast_Units"] = units.round().astype(int)
    team = _team()
    out["Team"] = team or ""
    saved_at = datetime.now(ZoneInfo("Europe/Istanbul")).isoformat(timespec="seconds")
    out["Saved_At"] = saved_at                       # when the file was saved (Istanbul time)
    out.to_csv(SUBMISSION / "forecast.csv", index=False)
    name = UPLOAD / f"{team or 'TeamXX'}_forecast.csv"
    shutil.copy(SUBMISSION / "forecast.csv", name)
    print(f"Saved: submission/forecast.csv (at {saved_at[11:16]}). The forecast that gets scored "
          "is the file you upload to the form by minute 45.")
    baseline = "lag12" in forecast and (out.Forecast_Units == forecast.lag12.round()).all()
    if download is None:
        download = not baseline                      # the benchmark forecast isn't downloaded
    if baseline and not download:
        print("Note: this is still the benchmark (seasonal naive) forecast. Run the cell again "
              "once your own model is in; on Colab the file is downloaded then.")
    if download and "google.colab" in sys.modules:
        from google.colab import files
        files.download(str(name))
        print(f"{name.name} has been downloaded to your computer. Upload it to the “Minute-45 "
              "forecast” form by minute 45 and pick your team in the form. The form (the one "
              "your instructor shows on screen): https://forms.gle/8aogMg4kcKzk1ipbA")


def plot_forecast(history, forecast):
    """Plots past demand and the January–March 2026 forecast by product on one chart (forecast
    dashed)."""
    past = history.pivot(index="Month", columns="Product", values="Demand_Units")
    ahead = forecast.pivot(index="Month", columns="Product", values="Forecast_Units")
    ahead = pd.concat([past.tail(1), ahead])         # start the dashed lines from December 2025
    past.index = pd.PeriodIndex(past.index, freq="M")
    ahead.index = pd.PeriodIndex(ahead.index, freq="M")
    ax = past.plot(figsize=(11, 3.5), title="Monthly demand and the January–March 2026 forecast "
                                            "(units)")
    colours = [line.get_color() for line in ax.get_lines()]
    ahead.plot(ax=ax, style="--", color=colours, legend=False)
    return ax

### Step 1.1 · The backtest

A model is a function that takes `(train, test)` and returns one forecast for every row of `test`, in the same order. `seasonal_naive` is the simplest example. Your job is to pick the right `train` and `test` rows for each origin, and to drop the `Demand_Units` column from `test` before the model sees it. The origins are given in the ground rules.

Once you've written it, run the check cell directly below.

In [ ]:
# the eight origins from the ground rules
ORIGINS = ["2023-12", "2024-03", "2024-06", "2024-09", "2024-12", "2025-03", "2025-06", "2025-09"]


def backtest(model):
    """Backtest. Returns one row per forecast: Month, Product, Demand_Units and Forecast."""
    results = []
    # TODO for each origin in ORIGINS:
    #        train = the rows the model can learn from at that origin
    #        test  = the rows it has to forecast (the model sees them without Demand_Units)
    #        forecast = model(train, test.drop(columns="Demand_Units"))
    #        append the test rows, with a Forecast column, to results
    if not results:
        return pd.DataFrame(columns=["Month", "Product", "Demand_Units", "Forecast"])
    return pd.concat(results)

In [ ]:
check_backtest(backtest)

### Step 1.2 · Your model

A few possible approaches (pick one, combine them or go your own way):

- **Exponential smoothing** (for example Holt-Winters in `statsmodels`) uses only the demand series and estimates a level, a trend and a twelve-month seasonal pattern.
- **A regression on log demand** can use the explanatory columns in the data. Which of them would a demand planner expect to matter?
- **Tree-based models** (`scikit-learn`) can use the same columns. With only 60 months per product, watch out for overfitting: check that they really generalise. Trees also can't extrapolate a trend beyond the data they were trained on, so consider removing the trend first.

Look at the bias for each product, not just the overall WAPE. A model can look fine in total and still be off in the same direction on one product, month after month.

In [ ]:
def my_model(train, test):
    """Your model. Returns one forecast per row of test, in the same order."""
    # TODO build your model here; until then it returns the benchmark
    return seasonal_naive(train, test)

In [ ]:
# add your other models here too
compare({"Seasonal naive": seasonal_naive, "My model": my_model}, backtest)

### Step 1.3 · The January–March 2026 forecast

Forecast the three months with your model and save the file. The cell downloads it to your computer as `TeamXX_forecast.csv`; upload that file to the “Minute-45 forecast” form **by minute 45**. That is the forecast that gets scored: changing it later, for example after the consensus plan comes out, doesn't count. You can upload again if you need to; the last file before minute 45 counts.

If the forecast is still the benchmark's (that is, `my_model` hasn't changed yet), nothing is downloaded. To download it as it is, use `save_forecast(forecast, download=True)`. The cell below plots the history and your forecast together; you can use the chart on the forecast and plan slide.

In [ ]:
future = plan_q1.copy()
# TODO prepare the future table you'll forecast from

forecast = future.assign(Forecast_Units=my_model(history, future))
save_forecast(forecast)
shown = ["Month", "Product", "Promo", "Planned_Price_EUR", "Forecast_Units"]
show(forecast[shown].round({"Forecast_Units": 0}))

In [ ]:
# past demand and your forecast, for the forecast and plan slide
plot_forecast(history, forecast)
plt.show()

## 2 · Production plan and takt (minutes 45–57)

**Goal:** turn the consensus demand into planned production and a takt time for each month, and decide which month the line has to be designed for.

**What you have:** the consensus demand and Deniz's email, unlocked with the password announced at minute 45; plus `calendar` (working days, shifts, breaks, maintenance, additional stops) and `products` (stock on 31 December, first-pass yield). From here on, everyone plans against the consensus demand.

In [ ]:
# READY · Planning tools

def unlock_release(file, password):
    """Unpacks a password-protected file into the release/ folder next to the notebook."""
    if not password:
        print("Type the password announced in the room and run the cell again.")
        return
    password = "-".join(password.lower().split())      # works with capitals or spaces too
    with zipfile.ZipFile(file) as z:
        for member in z.infolist():
            try:
                z.extract(member, ".", pwd=password.encode())
            except RuntimeError:
                print("Wrong password, try again.")
                return
            except (zipfile.BadZipFile, zlib.error):
                Path(member.filename).unlink(missing_ok=True)       # remove the half-written file
                print("Wrong password, try again.")
                return
        print("Unpacked:", ", ".join(n for n in z.namelist() if not n.endswith("/")))


BACKUP_FILES = ("consensus_demand.csv", "Deniz_Email_45_min.pdf",
                "urgent_order.csv", "Urgent_Email_95_min.pdf")


def use_backup_release():
    """If a protected file won't open: moves the unprotected files from your instructor (uploaded
    to Colab or next to the notebook) into release/."""
    RELEASE.mkdir(exist_ok=True)
    moved = []
    for folder in (Path.cwd(), Path.cwd().parent, CONTENT, CONTENT / "drive" / "MyDrive"):
        for file in BACKUP_FILES:
            if (folder / file).exists() and not (RELEASE / file).exists():
                shutil.move(str(folder / file), str(RELEASE / file))
                moved.append(file)
    print("Moved into release/:", ", ".join(moved) if moved else "—")


def load_consensus():
    """Returns the consensus demand as a {(month, product): units} dictionary."""
    if not (RELEASE / "consensus_demand.csv").exists():
        print("Unlock the minute-45 file first.")
        return {}
    demand = pd.read_csv(RELEASE / "consensus_demand.csv")
    show(demand.pivot(index="Month", columns="Product", values="Consensus_Demand_Units"))
    return {(r.Month, r.Product): r.Consensus_Demand_Units for r in demand.itertuples()}


def add_urgent(D):
    """Adds the URGENT order to the consensus demand D and returns the new dictionary."""
    if not D:
        print("Unlock the minute-45 file first.")
        return {}
    if not (RELEASE / "urgent_order.csv").exists():
        print("Unlock the minute-95 file first.")
        return dict(D)
    urgent = pd.read_csv(RELEASE / "urgent_order.csv")
    show(urgent)
    out = dict(D)
    for r in urgent.itertuples():
        out[(r.Month, r.Product)] += r.Additional_Units
    return out


def _mark(ok):
    return "✓" if ok else "✗"


# the placeholder product mix suggested in the introduction
INTRO_MIX = [0.4, 0.35, 0.25]


def _intro_mix(mix):
    """True if the product mix is still the placeholder from the introduction."""
    return bool(mix) and [mix.get(p) for p in PRODUCTS] == INTRO_MIX


def check_plan(avail, plan, takt=None, design_month=None, C=None, mix=None):
    """Compares your January numbers with the correct ones and checks that DESIGN_MONTH, C and MIX
    fit together."""
    avail, takt, mix = _as_dict(avail), _as_dict(takt) or {}, _as_dict(mix)
    if not avail or plan is None or len(plan) == 0:
        print("available_time() and production_plan() return nothing yet.")
        return
    jan_planned = plan[plan.Month == "2026-01"].Planned_Production.sum()
    ok = abs(avail.get("2026-01", 0) - JAN_AVAILABLE) < 1
    print(_mark(ok), f"January available time (correct: {fmt(JAN_AVAILABLE)} s)")
    ok = abs(jan_planned - JAN_PLANNED) <= 3
    print(_mark(ok), f"January planned production, all products (correct: {fmt(JAN_PLANNED)} "
                     "units)")
    if takt:
        ok = abs(takt.get("2026-01", 0) - JAN_TAKT) < 0.03
        print(_mark(ok), f"January takt time (correct: {fmt(JAN_TAKT, 2)} s)")
    if C is not None or mix:
        design_takt = takt.get(design_month)
        ok = (bool(mix) and abs(sum(mix.values()) - 1) < 1e-6
              and C is not None and design_takt is not None and abs(C - design_takt) < 0.01)
        print(_mark(ok), "MIX adds up to 1 and C equals the takt of DESIGN_MONTH (this doesn't "
                         "check which month you picked)")
    if _intro_mix(mix):
        print("⚠ MIX is still the placeholder suggested in the introduction. Use the product mix "
              "you worked out in Part 2.")

### Step 2.1 · Unlock the file

Type the password below and run the cell. Then read Deniz's email (`release/Deniz_Email_45_min.pdf`).

If the protected file won't open, your instructor will give you unprotected copies: drag them into the *Files* panel on the left-hand side of Colab and run `use_backup_release()`; the files are moved into the `release/` folder.

In [ ]:
unlock_release("release_45_min.zip", password="")   # ← type the password here
D = load_consensus()                                # {(month, product): consensus demand}

### Step 2.2 · Available time and planned production

The formula for available time is in the ground rules. Deniz's email explains how to work out planned production; keep the calculation notes in mind too.

In [ ]:
def available_time(calendar):
    """Returns each month's available time in seconds: {month: seconds}."""
    # TODO the available-time formula from the ground rules
    return {}


def production_plan(D):
    """Returns one row per month and product: Month, Product and Planned_Production."""
    # TODO the recipe in Deniz's email and the calculation notes
    return pd.DataFrame(columns=["Month", "Product", "Planned_Production"])


avail, plan = available_time(calendar), production_plan(D)

### Step 2.3 · Takt time and the design month

Work out the takt time for every month and decide which month to design the line for. Put that month in `DESIGN_MONTH`, its takt time in `C` and its product mix in `MIX` (the ground rules define the product mix).

Then run the check cell. It shows the correct January values; the other months are up to you. For `DESIGN_MONTH`, `C` and `MIX` it only checks that they fit together; it doesn't tell you whether you picked the right month.

In [ ]:
takt = {}                                     # {month: takt time in seconds}
# the design month, its takt time and its product mix {product: share}
DESIGN_MONTH, C, MIX = None, None, None
# TODO the takt time of every month; then DESIGN_MONTH, C and MIX

In [ ]:
check_plan(avail, plan, takt, DESIGN_MONTH, C, MIX)

## 3 · Line balancing (minutes 57–90)

**Goal:** a line that can run all quarter. Selin's email sets out what's expected; the rules the line has to follow are in Hasan's and Selin's messages.

**What you have**
- `TASKS`, `TIME[task][product]` (seconds), `GROUP[task]` (process group), `ARCS` (before–after pairs) and `CURRENT_LINE` (today's layout). The task names are in the `tasks` table.
- `weighted(mix)`: weighted task times. `station_loads(line, mix)`: each station's tasks, weighted load, time per product and process groups; `station_table(line, mix)` shows the same as a table. `capacity(line, mix, seconds)`: how many units a line can make in a given time.
- `check_assignment(line, C, mix)`: the referee. It only checks the basics: every task at exactly one station, the sequence kept, every station's load within takt, and stations numbered from 1 without gaps. The rules in the messages are your job.

In [ ]:
# READY · Line tools

def weighted(mix):
    """Weighted task times for a product mix {product: share}."""
    mix = dict(mix)
    return {i: sum(mix[p] * TIME[i][p] for p in PRODUCTS) for i in TASKS}


def station_loads(line, mix):
    """For each station: its tasks, weighted load, time per product and process groups, at full
    precision."""
    w, out = weighted(mix), {}
    for task, k in sorted(dict(line).items()):
        if task not in TIME:
            continue
        if k not in out:
            out[k] = {"Tasks": [], "Load": 0.0, "Groups": set(), **{p: 0 for p in PRODUCTS}}
        s = out[k]
        s["Tasks"].append(task)
        s["Load"] += w[task]
        s["Groups"].add(GROUP[task])
        for p in PRODUCTS:
            s[p] += TIME[task][p]
    return dict(sorted(out.items()))


def station_table(line, mix):
    """One row per station: tasks, weighted load, time per product and process groups (loads
    rounded for display)."""
    rows = {k: {"Tasks": ", ".join(s["Tasks"]), "Weighted": round(s["Load"], 2),
                **{p: s[p] for p in PRODUCTS}, "Groups": ", ".join(sorted(s["Groups"]))}
            for k, s in station_loads(line, mix).items()}
    return pd.DataFrame.from_dict(rows, orient="index")


def check_assignment(line, C, mix):
    """The referee: checks only the basics of a line {task: station}: every task at one station,
    the sequence, the takt and the station numbers. An empty list means no problem with these;
    it doesn't look at the rules in the messages."""
    line = dict(line)
    issues = [f"{i} isn't assigned to any station." for i in TASKS if i not in line]
    issues += [f"There's no task called {i}." for i in line if i not in TIME]
    issues += [f"Sequence broken: {a} (station {line[a]}) must come before {b} (station {line[b]})."
               for a, b in ARCS if a in line and b in line and line[a] > line[b]]
    stations = sorted(set(line.values()))
    if stations != list(range(1, len(stations) + 1)):
        issues.append("Stations must be numbered from 1 with no gaps (current numbers: "
                      f"{stations}).")
    for k, s in station_loads(line, mix).items():
        if s["Load"] > C + 1e-9:
            issues.append(f"Station {k}: {fmt(s['Load'], 2)} s > takt {fmt(C, 2)} s")
    return issues


def capacity(line, mix, seconds):
    """Units the line can make in the given time (seconds)."""
    loads = station_loads(line, mix)
    if not loads:
        return 0
    busiest = max(s["Load"] for s in loads.values())
    return math.floor(seconds / busiest)


def _own_rules(rules, line, C, mix):
    """Runs your rules function and returns what it found as a list."""
    found = rules(line, C, mix)
    if found is None:
        print("Note: check_rules didn't return a list (it returned None), so your rules weren't "
              "checked. The function should end with return problems.")
    return list(found or [])


def _report(line, C, mix, rules=None):
    issues = check_assignment(line, C, mix)
    print("Referee (basic rules):" + ("" if issues else " no problems"))
    for issue in issues:
        print("  -", issue)
    if rules is None:
        print("The referee skips the rules in the messages: pass rules=check_rules.")
        return
    problems = _own_rules(rules, line, C, mix)
    print("Your rules (check_rules):" + ("" if problems else " no problems found"))
    for problem in problems:
        print("  -", problem)


def show_current_line(C, mix, rules=None):
    """Shows the current line and asks the referee and, if given, your own rules."""
    mix = _as_dict(mix)
    if C is None or not mix:
        print("Set C and MIX in Part 2 first, or try placeholders for now: C, MIX = 85, "
              "{'ND100': 0.4, 'ND200': 0.35, 'ND300': 0.25}")
        return
    print("Current line, at your takt and mix:")
    show(station_table(CURRENT_LINE, mix))
    _report(CURRENT_LINE, C, mix, rules)


def report_line(line, C, mix, plan, takt, bound=None, rules=None):
    """Shows the stations, asks the referee and, if given, your own rules, and tests the line in
    all three months."""
    mix, takt = _as_dict(mix), _as_dict(takt)
    if C is None or not mix:
        print("Set C and MIX in Part 2 first, or try placeholders for now: C, MIX = 85, "
              "{'ND100': 0.4, 'ND200': 0.35, 'ND300': 0.25}")
        return
    if line is None:
        print("The model found no feasible line (balance_line returned None). Check your "
              "constraints and C.")
        return
    line = dict(line)
    if not line:
        print("No line yet: balance_line returns an empty dictionary.")
        return
    table = station_table(line, mix)
    show(table)
    lower = bound(C, mix) if bound else None
    print(f"Stations: {len(table)} | lower bound: {'—' if lower is None else lower} | "
          f"busiest station: {fmt(table.Weighted.max(), 2)} s | takt: {fmt(C, 2)} s")
    _report(line, C, mix, rules)
    if plan is not None and len(plan) and takt and all(m in takt for m in MONTHS):
        print("All three months, each at its own takt and mix:")
        for m in MONTHS:
            units = plan[plan.Month == m].set_index("Product").Planned_Production
            mix_m = (units / units.sum()).to_dict()
            issues = check_assignment(line, takt[m], mix_m)
            if rules is not None:
                issues += _own_rules(rules, line, takt[m], mix_m)
            lower = bound(takt[m], mix_m) if bound else None
            shown = "" if lower is None else f"lower bound {lower} | "
            print(" ", m, shown + ("no problems" if not issues else "; ".join(issues)))

### Step 3.1 · The line's rules, and today's line

Selin's and Hasan's messages contain rules for the line. Find them, and write them both into your model and into your own check. Your check is `check_rules(line, C, mix)`: it takes a line and returns the problems it finds as a list, which stays empty if there are none.

Try your check on today's line first: at your takt and mix, which station runs into trouble, and why?

In [ ]:
def check_rules(line, C, mix):
    """The line rules you found in the messages. Add a short description to problems for every case
    that breaks a rule; an empty list means no problems were found."""
    problems = []
    # TODO write a check for every rule you found in the messages
    #      (station_loads(line, mix): each station's tasks, load, product times and groups)
    return problems

In [ ]:
show_current_line(C, MIX, rules=check_rules)

### Step 3.2 · Lower bound and optimisation model

First, use `lower_bound` to work out the lower bound on the number of stations (the formula is in the prep note). Then build your model inside `balance_line`: decision variables, objective and constraints. The function should return the line as a `{task: station}` dictionary, with stations numbered from 1. The solver is ready for you: `model.solve(SOLVER)`.

As the ground rules ask, the model has to be **linear** and use a **single objective function**.

The next cell shows your line, asks the referee and your own `check_rules`, and tests the same layout in each of the three months, printing for each month the `lower_bound` at that month's own takt and mix (for the table on the forecast and plan slide). If even one month shows a problem, rethink your design.

In [ ]:
def lower_bound(C, mix):
    """Lower bound on the number of stations (prep note)."""
    # TODO
    return None


def balance_line(C, mix):
    """Builds and solves your line model. Returns {task: station}, or None if no line exists."""
    w = weighted(mix)                                  # weighted task times
    model = pulp.LpProblem("line", pulp.LpMinimize)

    # TODO decision variables

    # TODO objective (a single objective function)

    # TODO constraints: everything the line has to respect (read the messages again)

    model.solve(SOLVER)
    if pulp.LpStatus[model.status] != "Optimal":
        return None
    # TODO read each task's station from your variables
    return {}

In [ ]:
new_line = balance_line(C, MIX) if C is not None and MIX is not None else None
report_line(new_line, C, MIX, plan, takt, bound=lower_bound, rules=check_rules)

### Step 3.3 · If your model won't work: a fallback

`heuristic_line(C, MIX)` builds a line for you with a simple method called *ranked positional weight*. It fills one station at a time; at each step, of the tasks whose predecessors have all been assigned and that still fit, it picks the one with the largest positional weight: its own time plus the times of all the tasks that follow it.

It only looks at the sequence and the takt; it knows nothing about the rules in the messages. Check its line with `check_rules`. If you like, copy `heuristic_line` from the READY cell, together with the `fits` function inside it, into a cell of your own and add your rules to `fits`. There's also no guarantee it finds the fewest stations or the best balance: only an optimisation model can tell you that for sure.

In [ ]:
# READY · Fallback: the heuristic

def heuristic_line(C, mix):
    """Builds a line with the ranked positional weight method. Looks only at the sequence and the
    takt; no guarantee of the best line."""
    w = weighted(mix)
    successors = {i: [b for a, b in ARCS if a == i] for i in TASKS}
    predecessors = {i: [a for a, b in ARCS if b == i] for i in TASKS}

    def all_successors(i):
        found, stack = set(), list(successors[i])
        while stack:
            j = stack.pop()
            if j not in found:
                found.add(j)
                stack += successors[j]
        return found

    pos_weight = {i: w[i] + sum(w[j] for j in all_successors(i)) for i in TASKS}
    # assignments {task: station}, the station being filled and its tasks
    line, k, here = {}, 1, []

    def fits(i, here):
        """Can task i join the station being filled (here)?"""
        load = sum(w[j] for j in here + [i])
        return all(a in line for a in predecessors[i]) and load <= C + 1e-9

    while len(line) < len(TASKS):
        candidates = [i for i in TASKS if i not in line and fits(i, here)]
        if candidates:
            i = max(candidates, key=pos_weight.get)
            line[i] = k
            here.append(i)
        elif not here:                                # not even an empty station can take it
            raise ValueError("Some tasks don't fit at any station.")
        else:                                         # nothing fits: open a new station
            k, here = k + 1, []
    return line


# To use it, delete the # at the start of the two lines below:
# new_line = heuristic_line(C, MIX)
# report_line(new_line, C, MIX, plan, takt, bound=lower_bound, rules=check_rules)

## 4 · URGENT order (minutes 95–115)

**Goal:** a recommendation for the extra order, with its cost, and the point at which you'd change your mind.

The second password unlocks `urgent_order.csv` and Ece's email; the cell adds the order to the demand (`D_URGENT`).

**What we ask**
1. The new takt time, product mix and lower bound, and how many stations the order needs.
2. The prebuild option: how many units would you have to build early in January (`PREBUILD_UNITS`), and can January absorb them? `capacity(line, mix, seconds)` may come in handy.
3. What the two options cost (from Ece's table) and the spare capacity each one leaves: for the URGENT slide.
4. What you recommend and what would change your mind: Ece's question, the bottom box of the URGENT slide.

Whichever you recommend has to work on your own line: if you add stations, the new line has to pass both the referee and your own rules at the new takt and mix; if you prebuild, January has to be able to absorb it.

In [ ]:
unlock_release("release_95_min.zip", password="")   # ← type the password here
# {(month, product): demand including the order}
D_URGENT = add_urgent(D)

In [ ]:
C_URGENT, MIX_URGENT, urgent_line = None, None, None
PREBUILD_UNITS = None
STATION_COST_Q1_EUR, PREBUILD_COST_EUR = None, None
STATION_SPARE_UNITS, PREBUILD_SPARE_UNITS = None, None
# TODO new plan, takt (C_URGENT), mix (MIX_URGENT), lower bound and the line you need (urgent_line)
# TODO prebuild: put the units in PREBUILD_UNITS
# TODO costs (Ece's table): an extra station for the quarter, and the prebuild in total
# TODO spare capacity for the URGENT slide (capacity helps): how many units beyond what is needed
#   can
#      the new line make in February (extra station), and your line in January (prebuild)?

## 5 · Hand in (by minute 115)

Fill in your answers below and run the cell; it's worth running it once as soon as your line is ready, around minute 90. The cell writes `answers.csv` and the assignment files (`assignment_main.csv` and, if you add stations, `assignment_urgent.csv`) to `submission/` and lists anything still missing. On Colab, once every answer is filled in, two files are downloaded to your computer: `TeamXX_submission.zip` and the notebook itself, `TeamXX_notebook.ipynb` (if the browser asks whether to allow several downloads, say yes). To download with answers still missing, add `download=True` to the last line.

**By minute 115**, upload three files to the “Final submission” form: `TeamXX_submission.zip`, `TeamXX_slides.pptx` (save the slide template under that name) and `TeamXX_notebook.ipynb`. If the notebook doesn't download, use *File → Download → Download .ipynb* and rename it. If you upload again, your latest files count; nothing that arrives after minute 115 is accepted.

In [ ]:
# READY · Hand-in tool

DECISIONS = ("add_stations", "prebuild")


def _write_line(line, file):
    table = pd.DataFrame(sorted(dict(line).items()), columns=["Task", "Station"])
    table.to_csv(SUBMISSION / file, index=False)


def save_submission(answers, line, urgent_line=None, download=None):
    """Writes the answers and the assignments to submission/ and lists anything missing. On Colab
    it downloads TeamXX_submission.zip and TeamXX_notebook.ipynb once every answer is filled in,
    or if download=True is given."""
    answers, line, urgent_line = dict(answers), _as_dict(line), _as_dict(urgent_line)
    decision = answers.get("urgent_decision")
    if isinstance(decision, str):
        decision = answers["urgent_decision"] = decision.strip().lower()
    if decision is not None and decision not in DECISIONS:
        print(f"urgent_decision has to be 'add_stations' or 'prebuild' (you wrote {decision!r}); "
              "it was saved as empty.")
        answers["urgent_decision"] = decision = None
    team = _team()
    stamp = datetime.now(ZoneInfo("Europe/Istanbul")).isoformat(timespec="seconds")
    # the team number and the time are written into the answers file too
    rows = {**answers, "team": team or "", "saved_at": stamp}
    pd.DataFrame({"Key": list(rows), "Value": list(rows.values())}).to_csv(
        SUBMISSION / "answers.csv", index=False)
    # assignment files from an earlier save are removed
    for old in SUBMISSION.glob("assignment_*.csv"):
        old.unlink()
    if line:
        _write_line(line, "assignment_main.csv")
    if decision == "add_stations":
        if urgent_line:
            _write_line(urgent_line, "assignment_urgent.csv")
        else:
            print("urgent_decision is 'add_stations', but urgent_line is empty, so "
                  "assignment_urgent.csv was not written.")
    not_needed = {"prebuild_units"} if decision == "add_stations" else set()
    missing = [k for k, v in answers.items() if v is None and k not in not_needed]
    if _intro_mix({p: answers.get(f"mix_{p.lower()}") for p in PRODUCTS}):
        print("⚠ MIX is still the placeholder suggested in the introduction. Use the product mix "
              "you worked out in Part 2.")
    print("Files in submission/:", sorted(p.name for p in SUBMISSION.iterdir()))
    print("Answers still missing:", missing if missing else "—")
    if download is None:
        download = not missing                       # no download while answers are missing
    if "google.colab" not in sys.modules:
        return
    if not download:
        print("Some answers are still missing, so the submission file wasn't downloaded. To "
              "download it anyway: save_submission(answers, new_line, urgent_line, download=True)")
        return
    from google.colab import files
    name = f"{team or 'TeamXX'}_submission"
    shutil.make_archive(str(UPLOAD / name), "zip", SUBMISSION)
    files.download(str(UPLOAD / f"{name}.zip"))
    print(f"{name}.zip has been downloaded to your computer.")
    _download_notebook(team)
    print(f"Now, by minute 115, upload three files to the “Final submission” form: {name}.zip, "
          f"{team or 'TeamXX'}_slides.pptx and {team or 'TeamXX'}_notebook.ipynb. The form (the "
          "one your instructor shows on screen): https://forms.gle/zkj9uEyNLHyWTqFs9 · If you "
          "upload again, your latest files count.")


def _download_notebook(team):
    """On Colab, downloads the notebook itself as TeamXX_notebook.ipynb; if that fails, says how to
    do it by hand."""
    path = UPLOAD / f"{team or 'TeamXX'}_notebook.ipynb"
    try:
        from google.colab import _message, files
        notebook = _message.blocking_request("get_ipynb", timeout_sec=30)["ipynb"]
        path.write_text(json.dumps(notebook), encoding="utf-8")
        files.download(str(path))
        print(f"{path.name} has been downloaded to your computer.")
    except Exception:
        print("The notebook didn't download by itself: use File → Download → Download .ipynb and "
              f"rename the file {path.name}.")

In [ ]:
# the variables you've worked out come in by themselves; fill in whatever is still None
mix = dict(MIX) if MIX is not None else {}
answers = {
    "takt_jan": takt.get("2026-01"), "takt_feb": takt.get("2026-02"),
    "takt_mar": takt.get("2026-03"),
    "design_month": DESIGN_MONTH,                # the month you designed the line for (YYYY-MM)
    "planned_production_design": None,           # total planned production, design month (Step 2.2)
    "mix_nd100": mix.get("ND100"), "mix_nd200": mix.get("ND200"), "mix_nd300": mix.get("ND300"),
    "lower_bound_design": None,                  # lower bound at your design takt (Step 3.2)
    "stations": None,                            # number of stations you propose (Step 3.2)
    "current_bottleneck_load": None,             # current line's busiest station load (Step 3.1)
    "urgent_takt": C_URGENT,
    "urgent_lower_bound": None,                  # lower bound at the URGENT takt and mix (Part 4)
    "urgent_decision": None,                     # "add_stations" or "prebuild"
    # units to build early in January (may stay empty if you choose add_stations)
    "prebuild_units": PREBUILD_UNITS,
}
# TODO fill in the values that are None

save_submission(answers, new_line, urgent_line)